# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zezo-Elkafoury/Flyrank-internship-assignment-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the performance of a single content page for one reporting day within a specific month. Each row contains historical search and content signals that are available before a refresh decision is made.

This notebook uses February 2026 as the development window because it is a historical mid-panel month. Following the warehouse guidance, the final month is avoided during feature development to reduce the risk of leaking future information into the modeling process.

In [9]:
from google.colab import userdata
userdata.get('HF_TOKEN')
from datasets import load_dataset
df = load_dataset("FlyRank/internship-warehouse", data_files= "fact_content_daily_performance/month=2026-02/data_0.parquet")

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 89.0MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

In [12]:
import pandas as pd
sample_df = df['train'].to_pandas()
sample_df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-02-01,client_e547b89c05043229,content_7995404695ee1ffd,True,True,True,False,57.0,0.0,1778.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2026-02-01,client_e547b89c05043229,content_1eea820697c3b95a,True,True,True,False,13.0,0.0,85.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2026-02-01,client_e547b89c05043229,content_ccbb253f142217c3,True,True,True,True,59.0,0.0,1001.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2026-02-01,client_e547b89c05043229,content_ae16a6b9cf64c80a,True,True,True,False,17.0,0.0,287.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2026-02-01,client_e547b89c05043229,content_acf700633f016e5a,True,True,True,False,6.0,0.0,27.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
sample_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7355108 entries, 0 to 7355107
Data columns (total 30 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   report_date               object 
 1   client_hash_id            object 
 2   content_hash_id           object 
 3   client_has_gsc            bool   
 4   client_has_ga4            bool   
 5   gsc_data_available        object 
 6   ga4_data_available        object 
 7   gsc_impressions           float64
 8   gsc_clicks                float64
 9   gsc_sum_position          float64
 10  gsc_avg_position          float64
 11  ga4_pageviews             float64
 12  ga4_sessions              float64
 13  ga4_users                 float64
 14  ga4_engaged_sessions      float64
 15  ga4_total_engagement_sec  float64
 16  sessions_organic          float64
 17  sessions_direct           float64
 18  sessions_referral         float64
 19  sessions_social           float64
 20  sessions_paid           

In [14]:
sample_df['report_date'] = pd.to_datetime(sample_df['report_date'])
sample_df['month'] = sample_df['report_date'].dt.strftime('%Y-%m')
sample_df['month'].unique()

array(['2026-02'], dtype=object)

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Filter the development month (February 2026)
feb_df = sample_df[sample_df["month"] == "2026-02"]

# Verify the time window
print("Date range:")
print(f"Start: {feb_df['report_date'].min()}")
print(f"End:   {feb_df['report_date'].max()}")

print("\nNumber of rows:")
print(len(feb_df))

# Verify the grain (unit of analysis)
duplicates = feb_df.duplicated(
    subset=["client_hash_id", "content_hash_id", "report_date"]
).sum()

print("\nDuplicate (client, content, date) combinations:", duplicates)

if duplicates == 0:
    print("Each row represents one content page on one reporting day.")
else:
    print("Duplicate rows found. Check the dataset grain.")

Date range:
Start: 2026-02-01 00:00:00
End:   2026-02-28 00:00:00

Number of rows:
7355108

Duplicate (client, content, date) combinations: 0
Each row represents one content page on one reporting day.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- Features: gsc_impressions, gsc_clicks, gsc_sum_position, ga4_pageviews, gsc_avg_position

- Label: (Proxy Target) Refresh Priority Score. The warehouse does not contain a direct "refresh priority" label. Instead, the goal is to estimate a priority score from historical performance signals so that content pages can be ranked by their expected need for review.

- Context: report_date, client_hash_id, content_hash_id

- Excluded: client_hash_id (as a feature), content_hash_id (as a feature),
Any future observations, Any label-derived columns. They would not be available at the moment the refresh decision is made and would cause data leakage.




In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

#### Verify the grain (One row = one content page on one reporting day)

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check whether each (client, content, date) combination is unique

duplicates = feb_df.duplicated(
    subset=["client_hash_id", "content_hash_id", "report_date"]
).sum()

print(f"Duplicate rows: {duplicates}")

if duplicates == 0:
    print(" Verified: each row represents one content page on one reporting day.")
else:
    print(" Duplicate combinations found.")

Duplicate rows: 0
 Verified: each row represents one content page on one reporting day.


#### Verify the time window

In [18]:
print(f"Rows in February: {len(feb_df)}")
print(f"Start: {feb_df['report_date'].min()}")
print(f"End:   {feb_df['report_date'].max()}")

Rows in February: 7355108
Start: 2026-02-01 00:00:00
End:   2026-02-28 00:00:00


#### Verify Data availability

In [21]:
available_df = feb_df[
    (feb_df["gsc_data_available"] == True) &
    (feb_df["client_has_gsc"] == True)
]

print(f"Rows with GSC data available: {len(available_df)}")
print(f"Percentage: {len(available_df) / len(feb_df) * 100:.2f}%")

Rows with GSC data available: 2621783
Percentage: 35.65%


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- This dataset captures historical search performance but does not directly measure the business impact of refreshing a page. As a result, the Refresh Priority Score is a decision-support proxy rather than a direct measure of refresh success.

- The analysis uses a historical snapshot and therefore may not capture longer-term seasonal changes or future shifts in search behavior, and the model may need future enhancements

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.